<a href="https://colab.research.google.com/github/AmyMugeni/cognilens/blob/main/Cognilens_ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Path to your project in Drive
project_path = '/content/drive/MyDrive/Cognilens_MLtraining'
os.makedirs(project_path, exist_ok=True)
os.chdir(project_path)

print(f"Working directory set to: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory set to: /content/drive/MyDrive/Cognilens_MLtraining


## 1. Imports and Setup

In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score
import joblib

## 2. Generate Synthetic Dataset with Realistic Noise

In [7]:
np.random.seed(42)
num_samples = 5000

# Feature Inputs (Matching UsageStatsManager & User Profile)
mins_since_wake = np.random.randint(-60, 900, num_samples)
mins_before_bed = np.random.randint(-120, 900, num_samples)
session_duration_mins = np.random.exponential(scale=14, size=num_samples).astype(int) + 1
reopen_interval_mins = np.random.exponential(scale=25, size=num_samples).astype(int)
bsmas_score = np.random.randint(6, 31, num_samples)
is_schedule_conflict = np.random.choice([0, 1], size=num_samples, p=[0.6, 0.4])

df = pd.DataFrame({
    'mins_since_wake': mins_since_wake,
    'mins_before_bed': mins_before_bed,
    'session_duration_mins': session_duration_mins,
    'reopen_interval_mins': reopen_interval_mins,
    'bsmas_score': bsmas_score,
    'is_schedule_conflict': is_schedule_conflict
})

# Base Probability Matrix (Probabilistic instead of hard math to avoid 100% leakage)
def calculate_compulsive_prob(row):
    # Base probability
    prob = 0.15

    # Feature weights
    if row['session_duration_mins'] > 15: prob += 0.25
    if row['reopen_interval_mins'] < 5: prob += 0.30
    if row['bsmas_score'] > 20: prob += 0.15
    if row['is_schedule_conflict'] == 1: prob += 0.20

    # Bound between 0.05 and 0.95
    return min(max(prob, 0.05), 0.95)

# Generate probabilities and apply realistic human noise (10% random behavioral variance)
probs = df.apply(calculate_compulsive_prob, axis=1)
noise = np.random.normal(0, 0.08, num_samples) # Human behavior variance
final_probs = np.clip(probs + noise, 0, 1)

df['label'] = (final_probs > 0.50).astype(int)

print(f"Dataset Generated: {len(df)} samples")
print(f"Label Distribution:\n{df['label'].value_counts(normalize=True)}\n")

Dataset Generated: 5000 samples
Label Distribution:
label
0    0.6414
1    0.3586
Name: proportion, dtype: float64



## 3. Filter Out Hard Rule Windows & Prepare for ML

In [8]:
# Since Hard Rules run on-device BEFORE ML, we evaluate models ONLY on standard daytime hours
non_rule_mask = (df['mins_since_wake'] > 30) & (df['mins_before_bed'] > 0)
ml_df = df[non_rule_mask].copy()

X = ml_df[['session_duration_mins', 'reopen_interval_mins', 'bsmas_score', 'is_schedule_conflict']]
y = ml_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4. Train & Evaluate 3 Candidate Models

In [9]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
}

results = {}
best_model_name = None
best_f1 = -1.0
best_model_obj = None

print("=== BENCHMARKING MODELS (Standard Window Classifications) ===\n")

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {"Accuracy": acc, "F1-Score": f1, "ROC-AUC": auc}

    print(f"--- {name} ---")
    print(f"Accuracy : {acc * 100:.2f}%")
    print(f"F1-Score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}\n")

    # Selection criteria based on highest F1-Score (balances Precision & Recall)
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_model_obj = model

=== BENCHMARKING MODELS (Standard Window Classifications) ===

--- Random Forest ---
Accuracy : 88.92%
F1-Score : 0.8373
ROC-AUC  : 0.9579

--- Logistic Regression ---
Accuracy : 78.21%
F1-Score : 0.6679
ROC-AUC  : 0.8642

--- Gradient Boosting ---
Accuracy : 89.41%
F1-Score : 0.8501
ROC-AUC  : 0.9560



## 5. Declare Winner & Export Model (Joblib)

In [10]:
print(f"==================================================")
print(f" BEST PERFORMING MODEL: {best_model_name} (F1: {best_f1:.4f})")
print(f"==================================================")

# Save the winning model
output_filename = "cognilens_best_model.joblib"
joblib.dump(best_model_obj, output_filename)
print(f"Saved winner to '{output_filename}' ready for conversion to ONNX/TFLite.")

🏆 BEST PERFORMING MODEL: Gradient Boosting (F1: 0.8501)
Saved winner to 'cognilens_best_model.joblib' ready for conversion to ONNX/TFLite.


## 6. Convert to ONNX

In [14]:
# Install ONNX tools in Colab
!pip install skl2onnx onnxruntime -q

import onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Define input features: 4 float inputs (session_duration_mins, reopen_interval_mins, bsmas_score, is_schedule_conflict)
initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]

# Convert the winning model
onnx_model = convert_sklearn(best_model_obj, initial_types=initial_type, target_opset=11) # Specify opset version

# Save to ONNX file
with open("cognilens_gb_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Model successfully converted and saved as 'cognilens_gb_model.onnx'!")

Model successfully converted and saved as 'cognilens_gb_model.onnx'!


## 7. Interactive Model Classification

In [12]:
import numpy as np
import pandas as pd
import joblib

# Load the best performing model
best_model = joblib.load('cognilens_best_model.joblib')

print("Loaded model for interactive predictions.")

def predict_compulsion(session_duration_mins, reopen_interval_mins, bsmas_score, is_schedule_conflict):
    # Create a DataFrame for the single input, ensuring correct feature order
    input_data = pd.DataFrame([[session_duration_mins, reopen_interval_mins, bsmas_score, is_schedule_conflict]],
                              columns=['session_duration_mins', 'reopen_interval_mins', 'bsmas_score', 'is_schedule_conflict'])

    # Make prediction
    prediction = best_model.predict(input_data)[0]
    prediction_proba = best_model.predict_proba(input_data)[0][1] # Probability of the positive class (1)

    if prediction == 1:
        result = "Compulsive"
    else:
        result = "Not Compulsive"

    print(f"\n--- Prediction for new data ---")
    print(f"Input Features: Session Duration={session_duration_mins} mins, Re-open Interval={reopen_interval_mins} mins, BSMAS Score={bsmas_score}, Schedule Conflict={is_schedule_conflict}")
    print(f"Predicted Class: {result} (Probability of Compulsive: {prediction_proba:.2f})")
    return result

# --- Example Usage ---
print("\n--- Example 1: High compulsion risk ---")
predict_compulsion(
    session_duration_mins=30,      # Long session
    reopen_interval_mins=2,        # Rapid re-open
    bsmas_score=25,                # High BSMAS
    is_schedule_conflict=1         # During work/study hours
)

print("\n--- Example 2: Low compulsion risk ---")
predict_compulsion(
    session_duration_mins=5,      # Short session
    reopen_interval_mins=60,      # Long interval
    bsmas_score=10,               # Low BSMAS
    is_schedule_conflict=0        # During free time
)

# --- Your turn: Insert more data here ---
print("\n--- Your custom input ---")
# Modify these values to test different scenarios
predict_compulsion(
    session_duration_mins=15,
    reopen_interval_mins=10,
    bsmas_score=18,
    is_schedule_conflict=0
)


Loaded model for interactive predictions.

--- Example 1: High compulsion risk ---

--- Prediction for new data ---
Input Features: Session Duration=30 mins, Re-open Interval=2 mins, BSMAS Score=25, Schedule Conflict=1
Predicted Class: Compulsive (Probability of Compulsive: 0.99)

--- Example 2: Low compulsion risk ---

--- Prediction for new data ---
Input Features: Session Duration=5 mins, Re-open Interval=60 mins, BSMAS Score=10, Schedule Conflict=0
Predicted Class: Not Compulsive (Probability of Compulsive: 0.00)

--- Your custom input ---

--- Prediction for new data ---
Input Features: Session Duration=15 mins, Re-open Interval=10 mins, BSMAS Score=18, Schedule Conflict=0
Predicted Class: Not Compulsive (Probability of Compulsive: 0.01)


'Not Compulsive'

In [4]:
# Save the synthesized dataset to a CSV file
df.to_csv('synthesized_dataset.csv', index=False)
print("Synthesized dataset saved as 'synthesized_dataset.csv' in the current working directory.")

Synthesized dataset saved as 'synthesized_dataset.csv' in the current working directory.


In [ ]:
# Install ONNX tools in Colab
!pip install skl2onnx onnxruntime -q

import onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Define input features: 4 float inputs (session_duration_mins, reopen_interval_mins, bsmas_score, is_schedule_conflict)
initial_type = [('float_input', FloatTensorType([None, 4]))]

# Convert the winning model
onnx_model = convert_sklearn(best_model_obj, initial_types=initial_type)

# Save to ONNX file
with open("cognilens_gb_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Model successfully converted and saved as 'cognilens_gb_model.onnx'!")